In [ ]:
import os
import shutil
import subprocess

import numpy as np
import pandas as pd
import torch
import yaml

In [ ]:
PROJECT_ROOT = "sequifier-demo"
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
CONFIG_DIR = os.path.join(PROJECT_ROOT, "configs")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "outputs")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CONFIG_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
n_sequences = 100
seq_len = 128

np.random.seed(101)

records = []
for seq_id in range(n_sequences):
    start_num = np.random.randint(0, 5)
    for t in range(seq_len):
        val = ((start_num + t) % 5) + 1
        records.append(
            {
                "sequenceId": seq_id,
                "itemPosition": t,
                "targetToken": val,  # The categorical variable we want to predict
            }
        )

df = pd.DataFrame(records)
# Sequifier expects a specific format, usually csv or parquet
file_path = os.path.join(DATA_DIR, "synthetic-sequences.parquet")
df.to_parquet(file_path)

In [ ]:
df.head()

In [ ]:
def write_yaml(path, data):
    with open(path, "w") as f:
        yaml.dump(data, f, default_flow_style=False, sort_keys=False)

In [ ]:
preprocess_config = {
    "preprocessing_data_path": file_path,
    "read_format": "parquet",
    "write_format": "parquet",
    "selected_columns": ["targetToken"],
    "max_target_offset": 1,
    "window_stride": 1,
    "max_rows": None,
    "run": {"project_root": PROJECT_ROOT, "reproducibility": {"seed": 42}},
    "splits": {
        "train": {"window_length": 11, "fraction": 0.8},
        "validation": {"window_length": 11, "fraction": 0.1},
        "test": {"window_length": 11, "fraction": 0.1},
    },
}

pp_conf_path = os.path.join(CONFIG_DIR, "preprocess.yaml")
write_yaml(pp_conf_path, preprocess_config)

In [ ]:
train_config = {
    "model": {
        "backbone": {
            "architecture": {
                "model_dim": 16,
                "max_context_length": 10,
                "num_layers": 1,
                "attention": {"num_heads": 2},
                "feed_forward": {"hidden_dim": 32},
                "dropout": 0.0,
            }
        },
        "interfaces": {
            "default": {
                "input_columns": ["targetToken"],
                "target_columns": ["targetToken"],
                "ingestion": {
                    "type": "embedding",
                    "output_dim": 16,
                    "feature_embedding_dims": {"targetToken": 16},
                },
                "decoder": {"type": "linear", "prediction_length": 1},
            }
        },
        "execution": {"context_length": 10, "prediction_view": "causal"},
    },
    "run": {
        "project_root": PROJECT_ROOT,
        "model_name": "demo-cycle-model",
        "reproducibility": {"seed": 1010, "enforce_determinism": False},
        "execution": {
            "device": "cuda" if torch.cuda.is_available() else "cpu",
            "supervised": {
                "data_loading": {},
                "compilation": {"torch_compile": "none"},
                "precision": {},
            },
        },
        "recovery": {},
        "logging": {"log_interval_batches": 10000},
        "outputs": {},
        "export": {"prediction": {"onnx": {"batch_size": 10}}},
    },
    "datasets": {
        "default": {
            "sources": {
                "default": {
                    "metadata_config_path": "configs/metadata_configs/synthetic-sequences.json",
                    "read_format": "parquet",
                    "type": "prepared",
                }
            }
        }
    },
    "phases": {
        "supervised": {
            "algorithm": "supervised",
            "mode": "sequential",
            "sources": [{"source": "default.train"}],
            "batch_size": 32,
            "optimization": {
                "model": {
                    "learning_rate": 0.001,
                    "optimizer": {"name": "Adam", "arguments": {}},
                    "scheduler": {
                        "name": "StepLR",
                        "arguments": {"step_size": 1, "gamma": 0.99},
                    },
                }
            },
            "evaluation": {
                "sources": [{"source": "default.validation"}],
                "cadence": {"epochs": 1},
                "batch_size": 32,
                "before_phase": True,
                "on_snapshot": True,
            },
            "model_selection": {"handoff": "last"},
            "checkpoints": {"cadence": {"epochs": 5}},
            "learning": {
                "datasets": {
                    "default": {
                        "losses": {"targetToken": {"criterion": "CrossEntropyLoss"}}
                    }
                }
            },
            "budget": {"epochs": 5},
        }
    },
    "plan": ["supervised"],
}

tr_conf_path = os.path.join(CONFIG_DIR, "train.yaml")
write_yaml(tr_conf_path, train_config)

In [ ]:
infer_config = {
    "task": "forecast",
    "generation_steps": 5,
    "run": {
        "project_root": PROJECT_ROOT,
        "execution": {"device": "cpu", "batch_size": 10},
    },
    "artifact": {"paths": ["models/demo-cycle-model-best-5.onnx"]},
    "input": {"path": "data/synthetic-sequences-split2.parquet"},
    "output": {
        "format": "parquet",
        "output_probabilities": False,
        "decode_categories": True,
    },
}

inf_conf_path = os.path.join(CONFIG_DIR, "infer.yaml")
write_yaml(inf_conf_path, infer_config)

In [ ]:
subprocess.run(["sequifier", "preprocess", "--config-path", pp_conf_path], check=True)

In [ ]:
subprocess.run(["sequifier", "train", "--config-path", tr_conf_path], check=True)

In [ ]:
subprocess.run(["sequifier", "infer", "--config-path", inf_conf_path], check=True)

In [ ]:
pred_path = os.path.join(
    OUTPUT_DIR, "predictions", "demo-cycle-model-best-5", "part-000.parquet"
)

preds = pd.read_parquet(pred_path)

preds["targetToken"] = preds["targetToken"].astype(int)

print(f"Loaded predictions from: {pred_path}")
print(preds.head(3))

In [ ]:
seq_vals = preds["sequenceId"].to_numpy()
token_vals = preds["targetToken"].to_numpy()

# Check whether each generated sequence follows the learned 1, 2, 3, 4, 5 cycle.
sequence_id_change = np.concatenate([[0], (seq_vals[1:] != seq_vals[:-1]).astype(int)])
correct_increment = np.concatenate(
    [[1], (token_vals[1:] == token_vals[:-1] + 1).astype(int)]
)
last_was_five = np.concatenate([[1], (token_vals[:-1] == 5).astype(int)])
is_one = token_vals == 1

In [ ]:
# See what percentage of generated tokens conforms to the cycle rule.

mean_accuracy = (
    sequence_id_change | correct_increment | (last_was_five & is_one)
).mean()

print(f"{mean_accuracy * 100:.2f}% of generated tokens follow the cycle rule")

In [ ]:
print(f"Deleting demo project: {PROJECT_ROOT}")
shutil.rmtree(PROJECT_ROOT)